<a href="https://colab.research.google.com/github/Rajugithu/focusmeet/blob/master/Face.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os
import zipfile

# Correct file name
zip_file = "Face.zip"

# Extract the file
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall("Face")
    print("Extraction complete. Contents:")
    print(os.listdir("Face"))


Extraction complete. Contents:
['Train', 'Test', 'val']


In [3]:
!pip install tensorflow keras numpy matplotlib


In [4]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt


In [5]:
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [6]:
for layer in base_model.layers:
    layer.trainable = False


In [7]:
x = Flatten()(base_model.output)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
num_classes=5
output = Dense(num_classes, activation='softmax')(x)
model = Model(inputs=base_model.input, outputs=output)


In [8]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)
Train_datagen = ImageDataGenerator(rescale=1.0/255)
val_datagen = ImageDataGenerator(rescale=1.0/255)

train_generator = train_datagen.flow_from_directory(
    "/content/Face/Train",
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)
val_generator = val_datagen.flow_from_directory(
    "/content/Face/val",
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)
val_generator = val_datagen.flow_from_directory(
    "/content/Face/Test",
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)


Found 3500 images belonging to 5 classes.
Found 500 images belonging to 5 classes.
Found 1000 images belonging to 5 classes.


In [9]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [10]:
history = model.fit(
    train_generator,
    epochs=20,
    validation_data=val_generator
)


Epoch 1/20


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


110/110 ━━━━━━━━━━━━━━━━━━━━ 80s 559ms/step - accuracy: 0.2409 - loss: 1.8884 - val_accuracy: 0.3340 - val_loss: 1.3731
Epoch 2/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 52s 449ms/step - accuracy: 0.3342 - loss: 1.3993 - val_accuracy: 0.5390 - val_loss: 1.1773
Epoch 3/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 51s 445ms/step - accuracy: 0.3907 - loss: 1.3243 - val_accuracy: 0.5480 - val_loss: 1.0778
Epoch 4/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 53s 445ms/step - accuracy: 0.4249 - loss: 1.2758 - val_accuracy: 0.4730 - val_loss: 1.1748
Epoch 5/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 53s 459ms/step - accuracy: 0.4274 - loss: 1.2470 - val_accuracy: 0.5860 - val_loss: 1.0084
Epoch 6/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 58s 501ms/step - accuracy: 0.4316 - loss: 1.1993 - val_accuracy: 0.6210 - val_loss: 1.0207
Epoch 7/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 53s 456ms/step - accuracy: 0.4454 - loss: 1.1973 - val_accuracy: 0.6170 - val_loss: 0.9744
Epoch 8/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 82s 456ms/step - accuracy: 0.4537 - loss: 1.1598 - val

In [11]:
test_datagen = ImageDataGenerator(rescale=1.0/255)
test_generator = test_datagen.flow_from_directory(
    "/content/Face/Test",
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

loss, accuracy = model.evaluate(test_generator)
print(f"Test Accuracy: {accuracy*100:.2f}%")

Found 1000 images belonging to 5 classes.
32/32 ━━━━━━━━━━━━━━━━━━━━ 4s 132ms/step - accuracy: 0.6722 - loss: 0.8392
Test Accuracy: 68.40%
